<a href="https://colab.research.google.com/github/phineas-pta/gg_colab_AI_playground/blob/main/TTS_OmniVoice.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# TTS with OmniVoice

## get reference voice

In [ ]:
%%bash
pip install -q 'yt-dlp[default]' vietnormalizer

link=$( curl --silent https://api.github.com/repos/quickjs-ng/quickjs/releases/latest | grep 'qjs-linux-x86_64' | grep 'browser_download_url' | sed -E 's/.*(https:.*)"/\1/' )
wget -O qjs-linux-x86_64 $link &> /dev/null
chmod a+x qjs-linux-x86_64

In [ ]:
from yt_dlp import YoutubeDL

BASIC_CONFIG = { # ref: https://github.com/yt-dlp/yt-dlp/blob/master/yt_dlp/YoutubeDL.py
	"js_runtimes": {"quickjs": {"path": "/content/qjs-linux-x86_64"}},
	# "concurrent_fragment_downloads": 16,
	# "overwrites": True,
	"outtmpl": "%(id)s.%(ext)s",
	"format": "bestaudio/best",
	"postprocessors": [{"key": "FFmpegExtractAudio", "preferredcodec": "wav"}],
}

with YoutubeDL(BASIC_CONFIG | {"download_ranges": lambda *_: [{"start_time": 0., "end_time": 10.5}]}) as ydl:
	ydl.download("rZnygcVV3vI")

In [ ]:
from vietnormalizer import VietnameseNormalizer
NORMALIZER = VietnameseNormalizer()
TRANS_TABLE = str.maketrans("", "", "~#+")

def preprocess(text: str) -> str:
	text = text.translate(TRANS_TABLE)
	text = text.replace("\n\n", "\n")
	text = NORMALIZER.normalize(text)
	return text

## OmniVoice

In [ ]:
%pip install -q omnivoice

In [ ]:
import torch
import soundfile as sf
from omnivoice import OmniVoice

MODEL = OmniVoice.from_pretrained("kjanh/KhanhTTS-OmniVoice", device_map="cuda", dtype=torch.float16)

PROMPT = MODEL.create_voice_clone_prompt(
	ref_audio="rZnygcVV3vI.wav",
	ref_text="Kính thưa quý vị, từ bao nhiêu năm qua, tất cả các băng đọc truyện của Nguyễn Ngọc Ngạn do trung tâm Thuý Nga thực hiện, đã bị những người lạ đưa lên bừa bãi trên mạng.",
)

@torch.inference_mode()
def tts(text: str | list[str], wav_file: str | None = "output.wav") -> None:
	inputs = [text.strip("\n")] if isinstance(text, str) else [t.strip("\n") for t in text]
	outputs = MODEL.generate(text=inputs, language="vietnamese", voice_clone_prompt=PROMPT)
	if wav_file is not None:
		if isinstance(text, str):
			sf.write(wav_file, outputs[0], MODEL.sampling_rate)
		else:
			*name, ext = wav_file.split(".")
			basefilename = ".".join(name)
			for i, audio in enumerate(outputs):
				sf.write(f"{basefilename}_{i:04d}.{ext}", audio, MODEL.sampling_rate)

may help speed up inference in some cases but not always, see https://github.com/k2-fsa/OmniVoice/pull/46
```
torch._inductor.config.triton.cudagraph_skip_dynamic_graphs=True
MODEL.llm = torch.compile(MODEL.llm, mode="max-autotune", fullgraph=True, dynamic=True)
for _ in range(3): # warm-up (triggers compilation)
	tts(text="Xin chào.", wav_file=None)
```

In [ ]:
tts([
	"Mẹ tôi đang ngồi bên hiên nhà, vừa đan chiếc áo len màu xanh, vừa lắng nghe tiếng mưa rơi tí tách trên mái tôn.",
	"Tôi pha một ấm trà nóng, ngồi xuống cạnh cửa sổ và bắt đầu đọc nốt chương cuối cùng của cuốn sách mà mình yêu thích.",
	"Mặt trời vừa mới thức dậy đằng sau lũy tre xanh, tỏa những tia nắng vàng óng ả xuống con đường làng nhỏ bé, thân thuộc.",
	"Mọi người trong chợ đều nở nụ cười tươi tắn, chào hỏi nhau thân thiết như những người thân trong một gia đình lớn, đoàn kết.",
	"Ánh nắng buổi sáng len lỏi qua từng kẽ lá, tạo thành những vệt sáng trên mặt bàn gỗ của quán cà phê nhỏ."
])

## faster OmniVoice

In [ ]:
%pip install -q omnivoice-triton

In [ ]:
from omnivoice_triton import create_runner
import soundfile as sf

RUNNER = create_runner(name="hybrid", model_id="kjanh/KhanhTTS-OmniVoice")
RUNNER.load_model()

def tts_fast(text, wav_file="output.wav"):
	result = RUNNER.generate_voice_clone(
		text=text,
		ref_audio="rZnygcVV3vI.wav",
		ref_text="Kính thưa quý vị, từ bao nhiêu năm qua, tất cả các băng đọc truyện của Nguyễn Ngọc Ngạn do trung tâm Thuý Nga thực hiện, đã bị những người lạ đưa lên bừa bãi trên mạng.",
	)
	sf.write(wav_file, result["audio"], result["sample_rate"])

In [ ]:
tts_fast("Mẹ tôi đang ngồi bên hiên nhà, vừa đan chiếc áo len màu xanh, vừa lắng nghe tiếng mưa rơi tí tách trên mái tôn.")

In [ ]:
RUNNER.unload_model()